#Основы работы с Apache Spark


In [ ]:
# 1. Установка Java и Spark
#ДАННАЯ ЯЧЕЙКА ПЛОХО РАБОТАЕТ. УСТАНОВКА ПЕРЕНЕСЕНА НА СЛЕДУЮЩИЕ ЯЧЕЙКИ.
# Устанавливаем OpenJDK 21
!apt-get install openjdk-21-jdk-headless -qq > /dev/null

# Устанавливаем Spark 4.0.1
!wget -q https://archive.apache.org/dist/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz -O spark.tgz
!tar xf spark.tgz

# Устанавливаем findspark
!pip install -q findspark

# 2. Настройка окружения

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.0.1-bin-hadoop3"

# 3. Инициализация Spark

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Создаём SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("DataEngineeringCourse") \
    .getOrCreate()

# получаем SparkContext для RDD-заданий (5.1)
sc = spark.sparkContext

# 4. Проверка, что Spark работает

print("=" * 60)
print("Проверка Spark")
print("=" * 60)
print(f"Spark version: {spark.version}")
print(f"SparkContext: {sc}")
print(f"Master: {sc.master}")
print(f"App Name: {sc.appName}")
print(f"Default Parallelism: {sc.defaultParallelism}")

In [1]:
# Устанавливаем OpenJDK 21
!apt-get install -y openjdk-21-jdk-headless -qq > /dev/null

# PySpark через pip - без скачивания полного архива Spark
!pip install -q pyspark==4.0.1

In [2]:
#Проверка, что установилось
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [3]:
import os

# Настройка окружения
os.environ.pop("SPARK_HOME", None)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

print("SPARK_HOME =", os.environ.get("SPARK_HOME"))
print("JAVA_HOME  =", os.environ.get("JAVA_HOME"))

SPARK_HOME = None
JAVA_HOME  = /usr/lib/jvm/java-21-openjdk-amd64


In [4]:
from pyspark.sql import SparkSession
# Создаём SparkSession, получаем SparkContext для RDD-заданий (5.1), Проверка, что Spark работает
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("DataEngineeringCourse") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext

print("=" * 60)
print("Проверка Spark")
print("=" * 60)
print(f"Spark version:        {spark.version}")
print(f"SparkContext:         {sc}")
print(f"Master:               {sc.master}")
print(f"App Name:             {sc.appName}")
print(f"Default Parallelism:  {sc.defaultParallelism}")

Проверка Spark
Spark version:        4.0.1
SparkContext:         <SparkContext master=local[*] appName=DataEngineeringCourse>
Master:               local[*]
App Name:             DataEngineeringCourse
Default Parallelism:  2


In [5]:
#Загрузка датасета
from pyspark.sql.functions import col, avg, count, desc, round

games = spark.read.csv("/content/drive/MyDrive/Универ/Инжиниринг данных/variant_8_games.csv", header=True, inferSchema=True, quote='"', escape='"')
games.printSchema()
games.show(5, truncate=False)

root
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- price: double (nullable = true)
 |-- positive_ratio: integer (nullable = true)
 |-- release_date: date (nullable = true)

+------+------+----------+----------+-----+--------------+------------+
|app_id|name  |developer |genre     |price|positive_ratio|release_date|
+------+------+----------+----------+-----+--------------+------------+
|1     |Game_0|Ubisoft   |Simulation|30.14|94            |2017-05-15  |
|2     |Game_1|CD Projekt|Shooter   |0.05 |40            |2019-08-11  |
|3     |Game_2|CD Projekt|Strategy  |44.56|22            |2018-05-09  |
|4     |Game_3|CD Projekt|Strategy  |20.06|73            |2018-04-24  |
|5     |Game_4|Ubisoft   |Adventure |38.6 |97            |2023-08-27  |
+------+------+----------+----------+-----+--------------+------------+
only showing top 5 rows


In [6]:
#Rdd из row-объектов + вспомогательная функция
games_rdd = games.rdd
def rdd_show(rdd):
    for i in rdd.collect(): print(i)

In [7]:
#Задание 1: кол-во игр по разработчикам

dev_counts = (
    games_rdd
    .filter(lambda x: x['developer'] is not None)
    .map(lambda x: (x["developer"], 1))
    .reduceByKey(lambda x, y: x+y)
    .sortBy(lambda x: x[1], ascending=False)
)
rdd_show(dev_counts)

('Ubisoft', 186)
('EA', 181)
('Valve', 178)
('CD Projekt', 165)
('Square Enix', 150)
('Rockstar', 140)


In [10]:
#Задание 2: средняя цена и количество игр по жанрам
genre_count = (
    games_rdd
    .filter(lambda x: x["genre"] is not None)
    .map(lambda x: (x['genre'], 1))
    .reduceByKey(lambda x, y: x+y)
    )

genre_price_sum = (
    games_rdd
    .filter(lambda x: x['genre'] is not None and x['price'] is not None)
    .map(lambda r: (r['genre'], (float(r['price']), 1)))
    .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
    .map(lambda x: (x[0], x[1][0] / x[1][1] if x[1][1] > 0 else 0.0))
)

genre_stats = (
    genre_count
    .join(genre_price_sum)
    .map(lambda x: (x[0], x[1][1], x[1][0]))
    .sortBy(lambda x: x[2], ascending=False)
)

for genre, avg_price, cnt in genre_stats.collect():
  print(f"{genre}: сред. цена = {avg_price:.4f}, кол-во игр = {cnt}")

Shooter: сред. цена = 31.0038, кол-во игр = 182
Adventure: сред. цена = 30.1757, кол-во игр = 171
Action: сред. цена = 27.8308, кол-во игр = 167
Simulation: сред. цена = 29.3354, кол-во игр = 162
Strategy: сред. цена = 27.5059, кол-во игр = 159
RPG: сред. цена = 33.2657, кол-во игр = 159


In [11]:
# Альтернативное решение задания 2 через DataFrame API
genre_stats_df = (
    games
    .filter(col("genre").isNotNull() & col("price").isNotNull())
    .groupBy("genre")
    .agg(
        round(avg("price"), 4).alias("avg_price"),
        count("*").alias("games_count")
    )
    .orderBy(desc("games_count"))
)
genre_stats_df.show(truncate=False)

+----------+---------+-----------+
|genre     |avg_price|games_count|
+----------+---------+-----------+
|Shooter   |31.0038  |182        |
|Adventure |30.1757  |171        |
|Action    |27.8308  |167        |
|Simulation|29.3354  |162        |
|Strategy  |27.5059  |159        |
|RPG       |33.2657  |159        |
+----------+---------+-----------+



In [12]:
#Задание 3: топ 5 игр с наиб. positive_ratio
top5 = (
    games_rdd
    .filter(lambda r: r["positive_ratio"] is not None and r["name"] is not None)
    .map(lambda r: (float(r["positive_ratio"]), r["name"]))
    .sortByKey(ascending=False)
    .take(5)
)
for r in top5: print(r)
spark.stop()

(99.0, 'Game_84')
(99.0, 'Game_191')
(99.0, 'Game_373')
(99.0, 'Game_490')
(99.0, 'Game_496')


In [31]:
"""Задание 8.1: Загрузить variant_8_games.csv в RDD. Извлечь поле price. С
помощью aggregate() подсчитать сумму, количество, минимум и максимум
цен. Вычислить среднюю цену. Вывести (sum, count, min, max, avg).
"""
import math

prices_rdd = (
    games_rdd
    .filter(lambda x: x["price"] is not None)
    .map(lambda x: float(x["price"]))
)

zero = (0.0, 0, math.inf, -math.inf)
seq = lambda acc, p: (acc[0] + p, acc[1] + 1, min(acc[2], p), max(acc[3], p))
comb = lambda a, b: (a[0] + b[0], a[1] + b[1], min(a[2], b[2]), max(a[3], b[3]))

total_sum, cnt, mn, mx = prices_rdd.aggregate(zero, seq, comb)
avg_price = total_sum / cnt if cnt > 0 else 0.0

print(f"(Сумма, кол-во, мин., макс. и средняя) = ({total_sum:.4f}, {cnt:.4f}, {mn:.4f}, {mx:.4f}, {avg_price:.4f})")

(Сумма, кол-во, мин., макс. и средняя) = (29865.5300, 1000.0000, 0.0100, 59.9300, 29.8655)


In [15]:
"""
Задача 8.2. Извлечь поле developer. С помощью takeSample(False, 20)
получить 20 случайных игр. С помощью countByValue() подсчитать
распределение по разработчикам в выборке. Сравнить с распределением по
всему RDD. Объяснить, почему на малой выборке распределение может
отличаться.
"""
dev_rdd = (
    games_rdd
    .filter(lambda x: x['developer'] is not None)
    .map(lambda x: x['developer'])
)

sample_20 = dev_rdd.takeSample(False, 20)
#Распределение в выборке
sample_dist = sc.parallelize(sample_20).countByValue()

#Распределение по всему RDD
full_dist = dev_rdd.countByValue()
full_total = sum(full_dist.values())

print("Выборка из 20:")
for dev, c in sorted(sample_dist.items(), key=lambda x: -x[1]):
  print(f"{dev}: {c}")
print("\nВесь RDD (доля)")
for dev, c in sorted(full_dist.items(), key=lambda x: -x[1]):
  print(f"{dev}: {c}")

Выборка из 20:
EA: 7
Valve: 6
Square Enix: 2
Ubisoft: 2
CD Projekt: 2
Rockstar: 1

Весь RDD (доля)
Ubisoft: 186
EA: 181
Valve: 178
CD Projekt: 165
Square Enix: 150
Rockstar: 140


Распределения отличаются на малой выборке, т.к. takeSample - простая случайная выборка без возвращения: каждый элемент попадает в неё с вероятностью 20/N, независимо от других. Фактическое число попаданий разработчика колеблется вокруг ожидаемого 20*p_dev с отклонением sqrt(20p(1-p)).
TakeSample без seed каждый запуск даёт новый результат.

In [38]:
"""
Задача 8.3. Преобразовать строки CSV в PairRDD (name, positive_ratio). С
помощью top(5, key=lambda x: x[1]) получить топ-5 игр по positive_ratio.
Вывести первую игру через first(). С помощью takeSample(False, 5) получить 5
случайных игр. Для каждой из топ-5 вывести название и рейтинг через
foreach().
"""
name_ratio = (
    games_rdd
    .filter(lambda x: x['name'] is not None and x["positive_ratio"] is not None)
    .map(lambda x: (x['name'], int(x["positive_ratio"])))
)

top5 = name_ratio.top(5, key=lambda x: x[1])
print("Топ 5 по positive_ratio:")
for n, r in top5: print(f"{n}: {r}")

print(f"Первая: {name_ratio.first()}")

print("5 случайных:")
for n, r in name_ratio.takeSample(False, 5): print(f"{n}: {r}")

print("foreach():")
log_path = "/content/drive/MyDrive/Универ/Инжиниринг данных/foreach_top5.txt"

def log_part(iterator):
  with open(log_path, "a", encoding="utf-8") as f:
    for name, ratio in iterator: f.write(f"{name}: {ratio}\n")
sc.parallelize(top5).foreachPartition(log_part)
print(open(log_path, encoding="utf-8").read())
"""
print("\nforeach по топ-5:")
sc.parallelize(top5).foreach(lambda x: print(f"{x[0]}: {x[1]}"))"""

Топ 5 по positive_ratio:
Game_84: 99
Game_191: 99
Game_373: 99
Game_490: 99
Game_496: 99
Первая: ('Game_0', 94)
5 случайных:
Game_969: 63
Game_195: 57
Game_211: 78
Game_531: 15
Game_769: 34
foreach():
Game_373: 99
Game_490: 99
Game_496: 99
Game_84: 99
Game_191: 99



'\nprint("\nforeach по топ-5:")\nsc.parallelize(top5).foreach(lambda x: print(f"{x[0]}: {x[1]}"))'

In [29]:
"""
Задача 8.4. Преобразовать строки CSV в PairRDD (price, app_id). С
помощью sortBy(lambda x: x[0]) отсортировать по возрастанию цены. Вывести
топ-10 самых дешёвых. С помощью sortBy(lambda x: -x[0]) вывести топ-10
самых дорогих. Сравнить с takeOrdered(10, key=lambda x: x[0]) — результаты
должны совпадать.
"""
price_app = (
    games_rdd
    .filter(lambda x: x['price'] is not None and x["app_id"] is not None)
    .map(lambda x: (float(x['price']), int(x["app_id"])))
)

cheapest = price_app.sortBy(lambda x: x[0]).take(10)
print("Топ 10 самых дешёвых: ")
for p, a in cheapest: print(f"app_id={a}, price={p}")

expensive = price_app.sortBy(lambda x: -x[0]).take(10)
print("Топ 10 самых дорогих: ")
for p, a in expensive: print(f"app_id={a}, price={p}")

ordered = price_app.takeOrdered(10, key=lambda x: x[0])
print("Топ 10 TakeOrdered: ")
for p, a in ordered: print(f"app_id={a}, price={p}")

print("\nСовпадает с takeOrdered:", set(cheapest) == set(ordered))
print("Совпадает по порядку:", cheapest == ordered)

Топ 10 самых дешёвых: 
app_id=387, price=0.01
app_id=2, price=0.05
app_id=151, price=0.11
app_id=216, price=0.11
app_id=164, price=0.24
app_id=592, price=0.26
app_id=342, price=0.27
app_id=510, price=0.32
app_id=518, price=0.38
app_id=484, price=0.46
Топ 10 самых дорогих: 
app_id=53, price=59.93
app_id=776, price=59.91
app_id=761, price=59.87
app_id=700, price=59.82
app_id=180, price=59.76
app_id=650, price=59.71
app_id=913, price=59.7
app_id=809, price=59.58
app_id=914, price=59.57
app_id=277, price=59.52
Топ 10 TakeOrdered: 
app_id=387, price=0.01
app_id=2, price=0.05
app_id=151, price=0.11
app_id=216, price=0.11
app_id=164, price=0.24
app_id=592, price=0.26
app_id=342, price=0.27
app_id=510, price=0.32
app_id=518, price=0.38
app_id=484, price=0.46

Совпадает с takeOrdered: True
Совпадает по порядку: True


In [30]:
"""Задача 8.5. Сохранить топ-10 самых дорогих игр в /content/output/variant_8_expensive через coalesce(1).saveAsTextFile()."""
out_dir = "/content/output/variant_8_expensive"

!rm -rf /content/output
!mkdir -p /content/output

top10_expensive_rdd = sc.parallelize(expensive).coalesce(1).saveAsTextFile(out_dir)

!ls -la /content/output/variant_8_expensive
print("Содержимое: ")
!cat /content/output/variant_8_expensive/part-*

total 20
drwxr-xr-x 2 root root 4096 Nov  9 09:00 .
drwxr-xr-x 3 root root 4096 Nov  9 08:59 ..
-rw-r--r-- 1 root root  128 Nov  9 08:59 part-00000
-rw-r--r-- 1 root root   12 Nov  9 08:59 .part-00000.crc
-rw-r--r-- 1 root root    0 Nov  9 09:00 _SUCCESS
-rw-r--r-- 1 root root    8 Nov  9 09:00 ._SUCCESS.crc
Содержимое: 
(59.93, 53)
(59.91, 776)
(59.87, 761)
(59.82, 700)
(59.76, 180)
(59.71, 650)
(59.7, 913)
(59.58, 809)
(59.57, 914)
(59.52, 277)
